In [0]:
from	pyspark.sql	import	functions	as	F

In [0]:

base	=	"/Volumes/real_estate_dev/bronze/vl_real_estate"
dirty	=	(
    			spark.read.option("header",	True).option("inferSchema",	False)
				.csv(f"{base}/dirty_data_for_cleansing/properties_dirty.csv")
    )

valid_agents	=	spark.table("real_estate_dev.bronze.agents").select("agent_id")

valid_locations	=	spark.table("real_estate_dev.bronze.locations").select("location_id")

cleaned	=	(
    			dirty
				#	trim	whitespace	on	all	string	columns
				.withColumn("property_type",	F.trim(F.col("property_type")))
				#	standardize	property_type	casing/variants
				.withColumn("property_type",	F.upper(F.col("property_type")))
				.withColumn("property_type",	F.regexp_replace("property_type","SFH|SINGLE-FAMILY|SINGLE	FAMILY",	"SINGLE	FAMILY"))
				.withColumn("property_type",	F.regexp_replace("property_type","CONDOMINIUM",	"CONDO"))
				.withColumn("property_type",	F.regexp_replace("property_type","TOWN	HOUSE",	"TOWNHOUSE"))
				.withColumn("property_type",	F.initcap(F.col("property_type")))
    
				#	strip	$	and	,	from	listing_price	then	cast	to	double
				.withColumn("listing_price",	
                F.regexp_replace(F.col("listing_price"),	"[$,]",	"").cast("double"))
				
				#	strip	$	and	,	from	sold_price	then	cast	to	double
				.withColumn("sold_price",	
                F.regexp_replace(F.col("sold_price"),	"[$,]",	"").cast("double"))
				
				#	strip	$	and	,	from	hoa_fee	then	cast	to	double
				.withColumn("hoa_fee",	
                F.regexp_replace(F.col("hoa_fee"),	"[$,]",	"").cast("double"))
				
				#	cast	bathrooms,	square_feet,	and	lot_size_sqft	to	double
				.withColumn("bathrooms",	F.col("bathrooms").cast("double"))
				.withColumn("square_feet",	F.col("square_feet").cast("double"))
				.withColumn("lot_size_sqft",	F.col("lot_size_sqft").cast("double"))
    
				#	normalize	mixed	booleans
				.withColumn("has_pool",	F.when(F.upper(F.col("has_pool")).isin("YES",	"Y",	"1",	"TRUE"),True)
																													
                .when(F.upper(F.col("has_pool")).isin("NO",	"N",	"0",	"FALSE"),False)
				.otherwise(None))
				.withColumn("has_basement",	F.when(F.upper(F.col("has_basement")).isin("YES",	"Y",	"1",	"TRUE"),True)
																																	
                .when(F.upper(F.col("has_basement")).isin("NO",	"N",	"0",	"FALSE"),False)
				.otherwise(None))
				#	parse	mixed	date	formats
	            .withColumn("list_date",	F.coalesce(
								F.try_to_date("list_date",	"yyyy-MM-dd"),
						F.try_to_date("list_date",	"MM/dd/yyyy"),
						F.try_to_date("list_date",	"dd-MMM-yyyy"))
                         )
				
				#	handle	nulls	in	year_built	-	replace	with	year	from	list_date
				.withColumn("year_built",
					F.when(F.col("year_built").isNull(),	F.year(F.col("list_date")).cast("string"))
					.otherwise(F.col("year_built"))
				)
)
#------------------------------

#	quarantine	rows	that	fail	basic	validity	rules	instead	of	silently dropping them

#	parse	date	columns	to	handle	mixed	date	formats
cleaned_with_date	=	(cleaned
				.withColumn("updated_at",	F.coalesce(
						F.try_to_date("updated_at",	"yyyy-MM-dd"),
						F.try_to_date("updated_at",	"MM/dd/yyyy"),
						F.try_to_date("updated_at",	"dd-MMM-yyyy")
					)
				)
				.withColumn("created_at",	F.coalesce(
						F.try_to_date("created_at",	"yyyy-MM-dd"),
						F.try_to_date("created_at",	"MM/dd/yyyy"),
						F.try_to_date("created_at",	"dd-MMM-yyyy")
					)
				)
				.withColumn("sold_date",	F.coalesce(
						F.try_to_date("sold_date",	"yyyy-MM-dd"),
						F.try_to_date("sold_date",	"MM/dd/yyyy"),
						F.try_to_date("sold_date",	"dd-MMM-yyyy")
					)
				)
				
				#	handle	nulls	in	sold_price,	sold_date,	and	hoa_fee
				.withColumn("sold_price",
					F.when(F.col("sold_price").isNull(),	F.lit(0.0))
					.otherwise(F.col("sold_price"))
				)
				.withColumn("sold_date",
					F.when(F.col("sold_date").isNull(),	F.add_months(F.col("list_date"),	-120))
					.otherwise(F.col("sold_date"))
				)
				.withColumn("hoa_fee",
					F.when(F.col("hoa_fee").isNull(),	F.lit(0.0))
					.otherwise(F.col("hoa_fee"))
				)
				
				#	handle	nulls	in	bathrooms	based	on	property	type
				#	calculate	average	bathrooms	per	property	type
				.withColumn("avg_bathrooms_by_type",
					F.avg(F.col("bathrooms")).over(
						__import__("pyspark.sql.window",	fromlist=["Window"]).Window.partitionBy("property_type")
					)
				)
				.withColumn("bathrooms",
					F.when(F.col("bathrooms").isNull(),
						F.when(F.col("property_type") == "Single Family",	F.lit(1.0))
						.when(F.col("property_type") == "Land",	F.lit(0.0))
						.otherwise(F.col("avg_bathrooms_by_type"))
					)
					.otherwise(F.col("bathrooms"))
				)
				.drop("avg_bathrooms_by_type")
				
				#	handle	nulls	and	zeros	in	square_feet	based	on	property	type
				#	calculate	average	square_feet	per	property	type
				.withColumn("avg_sqft_by_type",
					F.avg(F.col("square_feet")).over(
						__import__("pyspark.sql.window",	fromlist=["Window"]).Window.partitionBy("property_type")
					)
				)
				.withColumn("square_feet",
					F.when((F.col("square_feet").isNull())	|	(F.col("square_feet")	==	0),	F.col("avg_sqft_by_type"))
					.otherwise(F.col("square_feet"))
				)
				.drop("avg_sqft_by_type")
				
				#	handle	nulls	and	zeros	in	lot_size_sqft	based	on	property	type
				#	calculate	average	lot_size_sqft	per	property	type
				.withColumn("avg_lot_size_by_type",
					F.avg(F.col("lot_size_sqft")).over(
						__import__("pyspark.sql.window",	fromlist=["Window"]).Window.partitionBy("property_type")
					)
				)
				.withColumn("lot_size_sqft",
					F.when((F.col("lot_size_sqft").isNull())	|	(F.col("lot_size_sqft")	==	0),
						#	for	apartments,	use	square_feet	instead	of	average
						F.when(F.col("property_type") == "Apartment",	F.col("square_feet"))
						.otherwise(F.col("avg_lot_size_by_type"))
					)
					.otherwise(F.col("lot_size_sqft"))
				)
				.drop("avg_lot_size_by_type")
				
				#	round	numeric	values	to	1	decimal	place
				.withColumn("square_feet",	F.round(F.col("square_feet"),	1))
				.withColumn("bathrooms",	F.round(F.col("bathrooms"),	1))
				.withColumn("listing_price",	F.round(F.col("listing_price"),	1))
				.withColumn("sold_price",	F.round(F.col("sold_price"),	1))
				.withColumn("hoa_fee",	F.round(F.col("hoa_fee"),	1))
				.withColumn("lot_size_sqft",	F.round(F.col("lot_size_sqft"),	1))
)

bad	=	cleaned_with_date.filter(
				F.col("listing_price").isNull()	|	(F.col("listing_price")	<=	0) | F.col("bedrooms").isNull()	|	
                (F.col("bedrooms").cast("double")	<	0)	| (F.col("year_built").cast("double")	>	2026)	|	F.col("list_date").isNull()
)

good	=	cleaned_with_date.subtract(bad)

#	drop	rows	with	orphaned	foreign	keys
good	=	(
    			good.join(valid_agents,	"agent_id",	"left_semi")
					.join(valid_locations,	"location_id",	"left_semi")
     )

#	de-duplicate	on	property_id,	keeping	the	most	recently	updated	row
w	=	F.row_number().over(__import__("pyspark.sql.window",	fromlist=["Window"]).Window
				.partitionBy("property_id").orderBy(F.col("updated_at").desc())
)

good	=	good.withColumn("rn",	w).filter("rn	=	1").drop("rn")

good.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("real_estate_dev.silver.properties_cleansed")
bad.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("real_estate_dev.silver.properties_quarantine")

print("Good	rows:",	good.count(),	"|	Quarantined	rows:",	bad.count())


In [0]:
%sql
select * from real_estate_dev.silver.properties_cleansed